In [ ]:
%pip install --upgrade agent-framework-redis

In [5]:
from agent_framework import InMemoryHistoryProvider
from agent_framework.azure import AzureOpenAIChatClient
from azure.identity import DefaultAzureCredential

agent = AzureOpenAIChatClient(credential=DefaultAzureCredential()).as_agent(
    name="MemoryBot",
    instructions="You are a helpful assistant.",
    context_providers=[InMemoryHistoryProvider("memory", load_messages=True)],
)

session = agent.create_session()

result = await agent.run("나는 채식주의 음식을 선호한다는 것을 기억해줘.", session=session)
print(result.text)
print("-"*60)
result = await agent.run("내가 어떤 음식을 선호하는지 기억하고 있어?", session=session)
print(result.text)
print("-"*60)
result = await agent.run("내 이름은 마이클이야. 내 이름도 기억해 줘", session=session)
print(result.text)
print("-"*60)

session2 = agent.create_session()

result = await agent.run("내 이름을 기억하고 있어?", session=session2)
print(result.text)




알겠습니다! 앞으로 당신이 채식주의 음식을 선호한다는 점을 기억하겠습니다. 채식 위주의 식단이나 레시피, 추천을 요청하실 때 참고하겠습니다. 언제든지 원하는 정보를 말씀해 주세요!
------------------------------------------------------------
네, 기억하고 있습니다! 당신은 채식주의 음식을 선호한다고 말씀하셨습니다. 앞으로 음식 추천이나 레시피가 필요하실 때 이를 참고해서 안내해드릴게요. 도움이 필요하시면 언제든 말씀해 주세요!
------------------------------------------------------------
알겠어요, 마이클! 앞으로 당신의 이름과 채식주의 음식 선호를 기억하고 도움을 드릴게요. 언제든 필요하신 게 있으면 말씀해 주세요!
------------------------------------------------------------
죄송하지만, 이전 대화 기록이나 당신의 이름을 기억할 수 없습니다. 다시 알려주시면 앞으로의 대화에서 사용할 수 있어요! 이름을 알려주시겠어요?


In [3]:
from agent_framework import Agent, InMemoryHistoryProvider
from agent_framework.azure import AzureOpenAIChatClient
from azure.identity import DefaultAzureCredential
from agent_framework.redis import RedisHistoryProvider

from typing import Annotated
from pydantic import Field
import uuid
from dotenv import load_dotenv

load_dotenv()

def get_weather(
    location: Annotated[str, Field(description="날씨 정보를 가져올 위치입니다. 📍")],
) -> str:
    """ 주어진 장소에 대한 날씨를 가져옵니다 """
    return f"{location}은 현재 구름이 많으며, 온도는 15°C 입니다."

redis_provider = RedisHistoryProvider(
    redis_url="redis://localhost:6379",
    source_id="redis_chat")

agent = Agent(
  client=AzureOpenAIChatClient(credential=DefaultAzureCredential()),
  instructions="You are a helpful weather agent.",
  tools=[get_weather],
  context_providers=[redis_provider])

session_id = str(uuid.uuid4())
session = agent.create_session(session_id=session_id)

print("사용자: 도쿄의 날씨는 어떤가요?")
response = await agent.run("도쿄의 날씨는 어떤가요?", session=session)
print(f"Agent: {response.text}")

print("\n사용자: 파리는 어떤가요?")
response = await agent.run("파리는 어떤가요?", session=session)
print(f"Agent: {response.text}")


# Phase 2: Simulate an application restart — reconnect using the same session ID in Redis
print("\n--- 2 단계 : Phase 2: 애플리케이션 '재시작' 후 재개한 상황 ---")

# 새로운 RedisHistoryProvider와 Agent를 생성합니다 (앱 재시작 시뮬레이션)
redis_provider2 = RedisHistoryProvider(
    redis_url="redis://localhost:6379",
    source_id="redis_chat")

agent2 = Agent(
  client=AzureOpenAIChatClient(credential=DefaultAzureCredential()),
  instructions="You are a helpful weather agent.",
  tools=[get_weather],
  context_providers=[redis_provider2])

# 동일한 session_id로 세션을 재연결하면 Redis에서 이전 대화 기록을 불러옵니다
session_resumed = agent2.create_session(session_id=session_id)

print("사용자: 내가 이전에 물어본 도시들의 날씨를 기억하고 있어?")
response = await agent2.run("내가 이전에 물어본 도시들의 날씨를 기억하고 있어?", session=session_resumed)
print(f"Agent: {response.text}")


사용자: 도쿄의 날씨는 어떤가요?
Agent: 도쿄는 현재 구름이 많으며, 기온은 15도입니다. 외출 시 참고하세요!

사용자: 파리는 어떤가요?
Agent: 파리는 현재 구름이 많으며, 기온은 15도입니다.

--- 2 단계 : Phase 2: 애플리케이션 '재시작' 후 재개한 상황 ---
사용자: 내가 이전에 물어본 도시들의 날씨를 기억하고 있어?
Agent: 네, 기억하고 있습니다!  
지금까지 아래 두 도시의 날씨를 물어보셨습니다.

- 도쿄: 구름이 많고 15°C
- 파리: 구름이 많고 15°C

다른 도시의 날씨도 궁금하신가요?
